# XAUUSD XGBoost Training

Run cells in order (▶). Estimated time: 30-60 min.

In [ ]:
# 1. Clone
!git clone https://github.com/MarcorpAI/tradermodel.git
%cd tradermodel

In [ ]:
# 2. Install deps
!pip install -e ".[train]" -q
print("Done")

---
## Get training data

**Pick ONE option below:**
- **Option A** — export from API (need a [free Twelve Data key](https://twelvedata.com/apikey))
- **Option B** — upload a zip from your computer

### Option A: Export from API

In [ ]:
API_KEY = input("Paste your Twelve Data API key: ")
import os
os.environ["TWELVE_DATA_API_KEY"] = API_KEY

print("Downloading XAUUSD M15/H1/H4 + EUR/USD + DXY...")
!python scripts/export_training_bundle.py --years 5 --output-dir data/training

print("\nDownloading US10Y yields...")
!python scripts/export_fred_series.py --output data/training/us10y_daily.csv

!ls -lh data/training/

### Option B: Upload zip

In [ ]:
from google.colab import files
uploaded = files.upload()
!mkdir -p data
!unzip -o *.zip -d data/
!ls -lh data/training/

---
## Validate, sanitize, train

In [ ]:
# 3. Validate files that exist
import os.path
required = [
    "data/training/xauusd_m15.csv",
    "data/training/xauusd_h1.csv",
    "data/training/xauusd_h4.csv",
    "data/training/eurusd_m15.csv",
    "data/training/dxy_m15.csv",
    "data/training/us10y_daily.csv",
]
for csv in required:
    if os.path.exists(csv):
        !python scripts/validate_training_data.py --csv "$csv"
    else:
        print(f"SKIP (not found): {csv}")

In [ ]:
# 4. Sanitize (fix high/low inversions in OHLC files only)
import glob
import pandas as pd
for f in sorted(glob.glob('data/training/*.csv')):
    cols = pd.read_csv(f, nrows=1).columns
    if all(c in cols for c in ["open","high","low","close"]):
        !python scripts/sanitize_training_data.py "$f"
    else:
        print(f"SKIP (non-OHLC): {f}")
print("Done")

In [ ]:
# 5. Check required files exist before training
import os, sys
required = ["xauusd_m15.csv", "xauusd_h1.csv", "xauusd_h4.csv", "eurusd_m15.csv"]
missing = [f for f in required if not os.path.exists(f"data/training/{f}")]
if missing:
    print(f"MISSING REQUIRED FILES: {missing}")
    print("Run Option A above to export them, or upload a complete zip.")
    raise SystemExit("Cannot train without all required CSVs")

print("All required files present. Starting training...")

In [ ]:
# 6. Train (~30-60 min)
# --target-mode binary: labels 0=SELL, 1=BUY based on future close +/- 0.5*ATR
# --binary: trains binary classifier with scale_pos_weight for BUY vs not-BUY
!python scripts/train_xgboost.py \
  --m15 data/training/xauusd_m15.csv \
  --h1 data/training/xauusd_h1.csv \
  --h4 data/training/xauusd_h4.csv \
  --dxy data/training/eurusd_m15.csv \
  --us10y data/training/us10y_daily.csv \
  --real-dxy data/training/dxy_m15.csv \
  --output models/xgb_xauusd_v1.pkl \
  --trials 50 \
  --target-mode binary \
  --binary

In [ ]:
# 7. Inspect
import joblib
a = joblib.load('models/xgb_xauusd_v1.pkl')
print(f"Type:        {a.get('artifact_type')}")
print(f"Features:    {len(a.get('feature_columns', []))} columns")
print(f"Model:       {type(a.get('model')).__name__}")

In [ ]:
# 8. Download
from google.colab import files
files.download('models/xgb_xauusd_v1.pkl')

---
## After download — on your machine

```bash
mv ~/Downloads/xgb_xauusd_v1.pkl models/overlap_macro_trend_xgb.pkl
python scripts/dry_model_signal.py --ignore-calendar
```